<a href="https://colab.research.google.com/github/HenryAdler/FlyRank/blob/main/Henry%2001_first_look_and_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Run it, then discover a real truth yourself

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/01_first_look_and_discovery.ipynb)

By the end of this notebook you will have:
1. **Run a real ML pipeline** on real (anonymized) search data and watched a learned model beat a hand-written rule.
2. **Rediscovered a real finding yourself** in ~10 lines of pandas.

No prior ML needed. Everything runs on the small anonymized dataset that ships with this repo — no credentials, no private data.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os
import pandas as pd
import numpy as np

# Garante que estamos na raiz do repositório
if os.path.basename(os.getcwd()) in ["notebooks", "work"]:
    while os.path.basename(os.getcwd()) != "FlyRank" and os.getcwd() != "/":
        os.chdir("..")

csv_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(csv_path), f"Arquivo não encontrado em {csv_path}. Verifique o diretório atual: {os.getcwd()}"

df = pd.read_csv(csv_path)
print(f"Dataset carregado: {df.shape[0]:,} páginas | {df['client_id'].nunique()} clientes | {df.shape[1]} colunas")

Dataset carregado: 30,000 páginas | 32 clientes | 44 colunas


## 1. My Lane and Why

**Selected Lane:** **Lane 1 — Content Refresh / Opportunity Scoring (Organic Search Decay)**

**Why this lane:**
In organic search operations, existing content libraries represent a substantial sunk cost, yet published pages naturally lose rankings, impressions, and click-through rate (CTR) over time as search intent shifts or competitors publish fresher material. Rather than treating every piece of content equally or relying solely on new content production, identifying which existing URLs are actively decaying—and still hold recoverable search visibility—offers a direct, measurable lever to protect and recover organic traffic across client portfolios.

You just ran a real ML system on real search data and saw a learned ranking beat a fixed rule. Now open `outputs/model_report.md` and skim it — that Markdown report is the *shape* of what your own capstone should produce.

## 2. The Question: Decision, Action, and Cost of a Wrong Call

* **Research Question:** Among published client pages with meaningful baseline visibility, which URLs are most likely to experience organic traffic/impression decay in the near term, and how should an editorial team prioritize them for content refresh versus retirement or passive monitoring?
* **Unit of Analysis:** An individual content URL (`content_id`) belonging to a specific client (`client_id`).
* **Model Output:** A calibrated decay risk score (probability of declining performance) paired with a tiered action queue (`Urgent Refresh`, `Standard Refresh`, `Review for Retirement`, `Monitor`).
* **Action Taken:** Content strategists and editors allocate weekly refresh capacity to the top-ranked recoverable pages (updating outdated facts, improving search intent alignment, and optimizing on-page elements), while routing low-visibility declining pages to consolidation/retirement review instead of full rewrites.
* **Cost of Getting It Wrong:**
  * **False Positive (flagging a healthy/stable page for refresh):** Wastes 3–6 hours of editorial and SEO specialist labor per page and risks disrupting rankings on a URL that was already performing well.
  * **False Negative (missing a high-value decaying page):** Results in silent, compounding organic traffic loss on high-impression pages where early intervention would have been much cheaper than regaining lost page-1 rankings from scratch.
* **Why Data / ML Helps Beyond a Static Rule:** Simple hand-written heuristics (such as sorting solely by raw word count or static keyword search volume) fail because decay is driven by interactions between current SERP position tier, recent trajectory (`trend_pct`), engagement signals, and content age. A supervised model trained with client-grouped validation (`client_holdout`) can weigh these multi-dimensional signals without overfitting to a single client's domain quirks.

## 3. Quick Look at the Data (Supporting Numbers)

Before building any predictive pipeline, we inspect the starter dataset (`data/raw/content_refresh_anonymized.csv`) to verify three empirical patterns that motivate this lane:
1. **Prevalence of decay:** How widespread is downward trajectory (`trend_direction == 'down'`) across the 30,000 URLs?
2. **The CTR cliff by position tier:** How sharply does CTR drop once a page slips out of `top_3` / `page_1` into `striking` distance (positions 11–20) or deeper?
3. **Why simple heuristics fail (Week-1 Discovery):** Checking whether keyword `search_volume` or `word_count` alone separates decaying pages from growing ones, and testing which `content_type` suffers the sharpest CTR drop within the same `position_tier`.

In [2]:
# 1. Distribuição das direções de tendência (prevalência de queda)
trend_dist = df["trend_direction"].value_counts(normalize=True).mul(100).round(1)
print("=== 1. Distribuição de Tendência (30.000 páginas) ===")
print(trend_dist.to_string())

# 2. Queda de CTR por faixa de posição (páginas visíveis com >= 100 impressões em 90d)
visible = df[df["impressions_90d"] >= 100].copy()
ctr_by_pos = visible.groupby("position_tier")["ctr"].agg(count="count", mean_ctr="mean").sort_values("mean_ctr", ascending=False)
ctr_by_pos["mean_ctr"] = ctr_by_pos["mean_ctr"].round(4)
print("\n=== 2. CTR Médio por Position Tier (impressions_90d >= 100) ===")
print(ctr_by_pos.to_string())

# 3. Descoberta da Semana 1: CTR por content_type dentro de um mesmo position_tier (ex: 'striking' e 'page_1')
print("\n=== 3. CTR Médio por Content Type dentro de 'page_1' e 'striking' ===")
ct_pos = (
    visible[visible["position_tier"].isin(["page_1", "striking"])]
    .groupby(["position_tier", "content_type"])["ctr"]
    .agg(pages="count", mean_ctr="mean")
    .round(4)
)
print(ct_pos.to_string())

# 4. Correlação entre search_volume e impressions_90d (apenas páginas com impressões > 0)
active = df[df["impressions_90d"] > 0]
corr_active = active["search_volume"].corr(active["impressions_90d"])
print(f"\n=== 4. Correlação search_volume vs impressions_90d (impressions > 0): {corr_active:.4f} ===")

=== 1. Distribuição de Tendência (30.000 páginas) ===
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8

=== 2. CTR Médio por Position Tier (impressions_90d >= 100) ===
               count  mean_ctr
position_tier                 
page_1          8633    0.3548
top_3            533    0.3341
striking        5903    0.2558
page_3_5        6058    0.1424
deep             879    0.0554

=== 3. CTR Médio por Content Type dentro de 'page_1' e 'striking' ===
                                  pages  mean_ctr
position_tier content_type                       
page_1        comparison article    226    0.1412
              feedly article        221    0.9048
              keyword article      8186    0.3458
striking      comparison article     76    0.1474
              feedly article         75    0.3580
              keyword article      5752    0.2559

=== 4. Correlação search_volume vs impressions_90d (impressions > 0): 0.0012 ===


## 4. Careful Words: What I Can and Can't Claim

* **Observed / Directional Associations Only:** The patterns computed above (such as lower CTR in deeper position tiers or near-zero linear correlation between static `search_volume` and `impressions_90d`) are *observational* properties of this historical snapshot. They do not prove causal mechanisms of Google's ranking algorithm.
* **No Guaranteed Ranking Recovery:** A high predicted decay score indicates that a page shares historical patterns with declining URLs; it does not guarantee that editing the page will restore traffic, as external SERP competition and seasonality also play a role.
* **Generalization Boundaries & Privacy:** All evaluations must use client-grouped splits (`client_id`) so the model is tested on unseen domains. No raw client identifiers or proprietary URLs will be exposed in public reports or portfolio artifacts.

## 5. Self-Check

- [x] **Lane chosen:** Predefined Lane 1 (Content Refresh / Opportunity Scoring).
- [x] **Decision & action named:** Prioritizing weekly editorial refresh queues vs. retirement/monitoring, with explicit false-positive and false-negative costs.
- [x] **Real starter numbers included:** Computed live from `data/raw/content_refresh_anonymized.csv` (decay prevalence, CTR by `position_tier`, `content_type` CTR breakdown, and active-page correlation).
- [x] **Beyond "train a model":** Explains why multi-signal ranking with client-holdout validation outperforms static rules.
- [x] **Careful language used:** Framed strictly as directional/observational without causal claims or client data exposure.